# Model Performance Analysis - Exoplanet Classification
This notebook provides a comprehensive analysis of a machine learning model for exoplanet habitability classification. It covers data loading, class distribution, model prediction evaluation, threshold optimization, and practical solutions for handling class imbalance. Visualizations and metric comparisons guide improvements for more effective detection of habitable planets.

In [ ]:
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tempfile
from pathlib import Path

import lifefinder.config as cfg 

sys.path.append('/app')

# Use tempfile to create a temporary directory
tmpdir = Path(tempfile.mkdtemp(prefix="lifefinder_eval_tests_"))
cfg.ROOT = tmpdir
cfg.DATA_DIR = tmpdir / "data"
cfg.EVALUATION_DIR = tmpdir / "evaluations"
cfg.MODELS_DIR = tmpdir / "models"

cfg.EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)



## Train a Simple Model

In [ ]:
from lifefinder.train import train as train_model

# Update configuration for testing
cfg.TARGET_FEATURE = 'habitable_zone_index'
cfg.FORCE_NASA_API_FETCH = False
cfg.TRAINING_CONFIG.update({
    "batch_size": 32,
    "epochs": 10,
    "learning_rate": 0.001,
    "hidden_dim": 128,
    "dropout": 0.1,
    "val_split": 0.2,
    "random_state": 42,
    "patience": 5,
    "hz_sigma": 0.8,
    "hz_threshold": 0.5,
})

train_result = train_model()

print(f"Best F1 Score: {train_result['best_f1']:.4f}")

model_path = train_result['best_model_path']
pipeline_path = train_result['best_pipeline_path']

## Load Data and Model

In [ ]:
from lifefinder.models.pytorch_classifier import ExoplanetNN
from lifefinder.models.trainer import Trainer

# Load new exoplanet data
raw_df = pd.read_csv('/app/data/raw/exoplanets.csv')
print(f"Input data shape: {raw_df.shape}")

# Load the preprocessing pipeline
pipeline = joblib.load(pipeline_path)

# Extract engineered features for X and target y
df_features = pipeline.named_steps["features"].transform(
    pipeline.named_steps["cleaning"].transform(raw_df)
)

# Extract target after feature engineering
y_true_raw = df_features[cfg.TARGET_FEATURE].values
# Convert continuous labels to binary using the same threshold as predictions
y_true = (y_true_raw > cfg.TRAINING_CONFIG["hz_threshold"]).astype(int)
# Remove target from X
X_df = df_features.drop(columns=[cfg.TARGET_FEATURE])

# Preprocess X
X = pipeline.named_steps["preprocessor"].transform(X_df)

# Load model
input_dim = X.shape[1]
exoplanetNN = ExoplanetNN(
    input_dim=input_dim,
    hidden_dim=cfg.TRAINING_CONFIG["hidden_dim"],
    dropout=cfg.TRAINING_CONFIG["dropout"],
)
trainer = Trainer(exoplanetNN)
trainer.load_checkpoint(model_path)


## Class Distribution Analysis

In [ ]:
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y_true, 
    test_size=cfg.TRAINING_CONFIG["val_split"],
    random_state=cfg.TRAINING_CONFIG["random_state"],
    stratify=y_true
)

print("Dataset splits:")
print(f"Train: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

# Generate predictions
trainer.model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test)
    y_proba = trainer.model(X_test_tensor).squeeze().numpy()
    y_pred = (y_proba >= 0.5).astype(int)

# Analyze class distribution
print("\n" + "="*60)
print("CLASS DISTRIBUTION ANALYSIS")
print("="*60)

# Overall dataset distribution
unique_total, counts_total = np.unique(y_true, return_counts=True)
print("\nFull Dataset:")
print(f"Not Habitable: {counts_total[0]} ({counts_total[0]/len(y_true):.1%})")
print(f"Habitable: {counts_total[1]} ({counts_total[1]/len(y_true):.1%})")
print(f"Imbalance Ratio: {counts_total[0]/counts_total[1]:.1f}:1")

# Test set distribution
unique_test, counts_test = np.unique(y_test, return_counts=True)
print("\nTest Set:")
print(f"Not Habitable: {counts_test[0]} ({counts_test[0]/len(y_test):.1%})")
print(f"Habitable: {counts_test[1]} ({counts_test[1]/len(y_test):.1%})")

# Visualize class distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Full dataset distribution
axes[0,0].bar(['Not Habitable', 'Habitable'], counts_total, 
             color=['#e74c3c', '#2ecc71'], alpha=0.8)
axes[0,0].set_title('Full Dataset Class Distribution')
axes[0,0].set_ylabel('Count')
for i, count in enumerate(counts_total):
    axes[0,0].text(i, count + max(counts_total)*0.01, str(count), 
                   ha='center', fontweight='bold')

# Test set distribution
axes[0,1].bar(['Not Habitable', 'Habitable'], counts_test,
             color=['#e74c3c', '#2ecc71'], alpha=0.8)
axes[0,1].set_title('Test Set Class Distribution')
axes[0,1].set_ylabel('Count')
for i, count in enumerate(counts_test):
    axes[0,1].text(i, count + max(counts_test)*0.01, str(count), 
                   ha='center', fontweight='bold')

# Pie chart for proportions
axes[1,0].pie(counts_total, labels=['Not Habitable', 'Habitable'], 
             autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'])
axes[1,0].set_title('Class Proportion')

# Habitability zone index distribution
axes[1,1].hist(y_true_raw, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[1,1].axvline(x=cfg.TRAINING_CONFIG["hz_threshold"], color='red', 
                 linestyle='--', linewidth=2, label=f'Threshold ({cfg.TRAINING_CONFIG["hz_threshold"]})')
axes[1,1].set_xlabel('Habitability Zone Index')
axes[1,1].set_ylabel('Frequency')
axes[1,1].set_title('Habitability Zone Index Distribution')
axes[1,1].legend()

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("INITIAL MODEL PERFORMANCE ON TEST SET")
print("="*60)
print(f"Test samples: {len(y_test)}")
print(f"Positive samples: {y_test.sum()}")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")

print("\nKey Issue Identified:")
imbalance_ratio = counts_test[0] / counts_test[1]
print(f"🔍 Severe class imbalance: {imbalance_ratio:.1f}:1 ratio")
print(f"🔍 Only {y_test.mean():.1%} of samples are habitable planets")
print(f"🔍 Model predictions: {y_pred.sum()} positive predictions")

## Prediction Distribution and Threshold Analysis

In [ ]:
from sklearn.metrics import precision_recall_curve, confusion_matrix, average_precision_score
from sklearn.metrics import roc_auc_score

# Analyze prediction probability distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Probability distribution by true class
axes[0,0].hist(y_proba[y_test == 0], bins=30, alpha=0.7, label='Not Habitable', 
               color='#e74c3c', density=True)
axes[0,0].hist(y_proba[y_test == 1], bins=30, alpha=0.7, label='Habitable', 
               color='#2ecc71', density=True)
axes[0,0].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Current Threshold')
axes[0,0].set_xlabel('Predicted Probability')
axes[0,0].set_ylabel('Density')
axes[0,0].set_title('Probability Distribution by True Class')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Box plot of probabilities
prob_data = [y_proba[y_test == 0], y_proba[y_test == 1]]
box_plot = axes[0,1].boxplot(prob_data, tick_labels=['Not Habitable', 'Habitable'], patch_artist=True)
box_plot['boxes'][0].set_facecolor('#e74c3c')
box_plot['boxes'][1].set_facecolor('#2ecc71')
axes[0,1].axhline(y=0.5, color='black', linestyle='--', alpha=0.7, label='Current Threshold')
axes[0,1].set_ylabel('Predicted Probability')
axes[0,1].set_title('Probability Distribution Summary')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
im = axes[0,2].imshow(cm, interpolation='nearest', cmap='Blues')
axes[0,2].set_title('Confusion Matrix (Threshold=0.5)')
tick_marks = np.arange(2)
axes[0,2].set_xticks(tick_marks)
axes[0,2].set_yticks(tick_marks)
axes[0,2].set_xticklabels(['Not Habitable', 'Habitable'])
axes[0,2].set_yticklabels(['Not Habitable', 'Habitable'])
axes[0,2].set_ylabel('True Label')
axes[0,2].set_xlabel('Predicted Label')

# Add text annotations to confusion matrix
thresh = cm.max() / 2.
for i, j in [(0,0), (0,1), (1,0), (1,1)]:
    axes[0,2].text(j, i, format(cm[i, j], 'd'),
                  ha="center", va="center",
                  color="white" if cm[i, j] > thresh else "black",
                  fontsize=16, fontweight='bold')

# Threshold optimization - F1 Score
thresholds = np.linspace(0.01, 0.99, 100)
f1_scores = []
recalls = []
precisions = []

for thresh in thresholds:
    y_pred_thresh = (y_proba >= thresh).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_thresh))
    recalls.append(recall_score(y_test, y_pred_thresh))
    precisions.append(precision_score(y_test, y_pred_thresh, zero_division=0))

best_f1_idx = np.argmax(f1_scores)
best_f1_threshold = thresholds[best_f1_idx]
best_f1_score = f1_scores[best_f1_idx]

axes[1,0].plot(thresholds, f1_scores, 'g-', linewidth=2, label='F1-Score')
axes[1,0].plot(thresholds, recalls, 'r--', linewidth=2, label='Recall')
axes[1,0].plot(thresholds, precisions, 'b--', linewidth=2, label='Precision')
axes[1,0].axvline(x=0.5, color='black', linestyle=':', alpha=0.7, label='Current (0.5)')
axes[1,0].axvline(x=best_f1_threshold, color='green', linestyle='-', alpha=0.8, 
                 label=f'Best F1 ({best_f1_threshold:.3f})')
axes[1,0].set_xlabel('Threshold')
axes[1,0].set_ylabel('Score')
axes[1,0].set_title(f'Metrics vs Threshold (Best F1: {best_f1_score:.3f})')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Precision-Recall Curve
precision_pr, recall_pr, thresholds_pr = precision_recall_curve(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

axes[1,1].plot(recall_pr, precision_pr, 'b-', linewidth=2, label=f'PR Curve (AUC={pr_auc:.3f})')
baseline_precision = y_test.sum() / len(y_test)
axes[1,1].axhline(y=baseline_precision, color='red', linestyle='--', 
                 label=f'Baseline ({baseline_precision:.3f})')
axes[1,1].set_xlabel('Recall')
axes[1,1].set_ylabel('Precision')
axes[1,1].set_title('Precision-Recall Curve')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# Performance comparison at different thresholds
thresholds_to_compare = [0.1, 0.3, 0.5, best_f1_threshold, 0.7]
comparison_metrics = []

for thresh in thresholds_to_compare:
    y_pred_comp = (y_proba >= thresh).astype(int)
    comparison_metrics.append({
        'Threshold': thresh,
        'Precision': precision_score(y_test, y_pred_comp, zero_division=0),
        'Recall': recall_score(y_test, y_pred_comp),
        'F1': f1_score(y_test, y_pred_comp),
        'Predictions': y_pred_comp.sum()
    })

comparison_df = pd.DataFrame(comparison_metrics)

x_pos = np.arange(len(thresholds_to_compare))
width = 0.25

axes[1,2].bar(x_pos - width, comparison_df['Precision'], width, label='Precision', alpha=0.8, color='blue')
axes[1,2].bar(x_pos, comparison_df['Recall'], width, label='Recall', alpha=0.8, color='red')
axes[1,2].bar(x_pos + width, comparison_df['F1'], width, label='F1-Score', alpha=0.8, color='green')

axes[1,2].set_xlabel('Threshold')
axes[1,2].set_ylabel('Score')
axes[1,2].set_title('Performance Comparison')
axes[1,2].set_xticks(x_pos)
axes[1,2].set_xticklabels([f'{t:.2f}' for t in thresholds_to_compare])
axes[1,2].legend()
axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Detailed analysis summary
print("\n" + "="*80)
print("                       DETAILED PREDICTION ANALYSIS")
print("="*80)

print("\n📊 PROBABILITY DISTRIBUTION INSIGHTS:")
print(f"   • Model output range: [{y_proba.min():.4f}, {y_proba.max():.4f}]")
print(f"   • Mean probability for habitable planets: {y_proba[y_test == 1].mean():.4f}")
print(f"   • Mean probability for non-habitable planets: {y_proba[y_test == 0].mean():.4f}")
# Assess separation between classes based on probability distributions
mean_hab = y_proba[y_test == 1].mean() if (y_test == 1).sum() > 0 else 0
mean_nonhab = y_proba[y_test == 0].mean() if (y_test == 0).sum() > 0 else 0
std_hab = y_proba[y_test == 1].std() if (y_test == 1).sum() > 0 else 0
std_nonhab = y_proba[y_test == 0].std() if (y_test == 0).sum() > 0 else 0

if abs(mean_hab - mean_nonhab) > (std_hab + std_nonhab) * 0.5:
    print(f"   • Model shows good separation between classes (mean diff: {mean_hab - mean_nonhab:.3f})")
elif abs(mean_hab - mean_nonhab) > (std_hab + std_nonhab) * 0.2:
    print(f"   • Model shows moderate separation between classes (mean diff: {mean_hab - mean_nonhab:.3f})")
else:
    print(f"   • Model shows poor separation between classes (mean diff: {mean_hab - mean_nonhab:.3f})")

print("\n🎯 THRESHOLD OPTIMIZATION RESULTS:")
print(f"   • Current threshold (0.5): F1={f1_score(y_test, y_pred):.3f}, Recall={recall_score(y_test, y_pred):.3f}")
print(f"   • Optimal threshold ({best_f1_threshold:.3f}): F1={best_f1_score:.3f}")

# Find threshold for 80% recall
recall_80_threshold = None
for i, thresh in enumerate(thresholds):
    if recalls[i] >= 0.8:
        recall_80_threshold = thresh
        break

if recall_80_threshold:
    y_pred_80_recall = (y_proba >= recall_80_threshold).astype(int)
    precision_80 = precision_score(y_test, y_pred_80_recall, zero_division=0)
    f1_80 = f1_score(y_test, y_pred_80_recall)
    print(f"   • For 80% recall ({recall_80_threshold:.3f}): Precision={precision_80:.3f}, F1={f1_80:.3f}")

print("\n💡 KEY FINDINGS:")

roc_auc = roc_auc_score(y_test, y_proba)
num_habitable = y_test.sum()
num_pred_habitable = y_pred.sum()
missed_habitable = int(num_habitable - num_pred_habitable)
f1_improvement = best_f1_score - f1_score(y_test, y_pred)

# Discrimination
if roc_auc > 0.9:
    print(f"   ✅ Model has excellent discrimination (ROC-AUC: {roc_auc:.3f})")
elif roc_auc > 0.8:
    print(f"   ✅ Model has good discrimination (ROC-AUC: {roc_auc:.3f})")
elif roc_auc > 0.7:
    print(f"   ⚠️  Model has moderate discrimination (ROC-AUC: {roc_auc:.3f})")
else:
    print(f"   ❌ Model has poor discrimination (ROC-AUC: {roc_auc:.3f})")

# Precision-recall trade-off
if best_f1_score > 0.7:
    print("   ✅ Good precision-recall trade-off available")
elif best_f1_score > 0.5:
    print("   ⚠️  Precision-recall trade-off is moderate")
else:
    print("   ❌ Precision-recall trade-off is poor")

# Threshold conservativeness
if num_pred_habitable < num_habitable * 0.5:
    print("   ⚠️  Current threshold is too conservative")
elif num_pred_habitable > num_habitable * 1.5:
    print("   ⚠️  Current threshold is too permissive")
else:
    print("   ✅ Current threshold is balanced")

# Missed habitable planets
if missed_habitable > 0:
    print(f"   ⚠️ Missing {missed_habitable}/{int(num_habitable)} habitable planets with current threshold")
else:
    print("   ✅ No habitable planets missed with current threshold")

# F1 improvement
if f1_improvement > 0.05:
    print(f"   🎯 Lowering threshold to {best_f1_threshold:.3f} would improve F1 by {f1_improvement:.3f}")
elif f1_improvement > 0.01:
    print(f"   🎯 Lowering threshold to {best_f1_threshold:.3f} would slightly improve F1 by {f1_improvement:.3f}")
else:
    print("   ✅ Current threshold is near optimal for F1")

print("\n📋 THRESHOLD COMPARISON TABLE:")
print(comparison_df.round(3).to_string(index=False))

## Data Leakage and Overfitting Detection

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Calculate y_pred_optimal using the best threshold from previous cell
y_pred_optimal = (y_proba >= best_f1_threshold).astype(int)

print("🔍 DATA LEAKAGE AND OVERFITTING DETECTION")
print("="*60)

# Check for data leakage indicators
print("\n1️⃣ DATA LEAKAGE INDICATORS:")

# Check if we're using the same data for train/test that was used for model training
print(f"   • Using same dataset split as training: {'⚠️  YES - POTENTIAL LEAKAGE' if cfg.TRAINING_CONFIG['random_state'] == 42 else '✅ NO'}")
print(f"   • Random state matches training: {cfg.TRAINING_CONFIG['random_state']} ({'⚠️  SUSPICIOUS' if cfg.TRAINING_CONFIG['random_state'] == 42 else '✅ OK'})")

# Check for perfect or near-perfect metrics
roc_auc = roc_auc_score(y_test, y_proba)
if roc_auc > 0.99:
    print("   ⚠️  ROC-AUC > 0.99 - HIGHLY SUSPICIOUS of data leakage")
elif roc_auc > 0.95:
    print("   ⚠️  ROC-AUC > 0.95 - Could indicate data leakage")
else:
    print(f"   ✅ ROC-AUC = {roc_auc:.3f} - Within reasonable range")

# Check prediction distribution - leakage often shows bimodal distributions
print(f"   • Prediction range: [{y_proba.min():.4f}, {y_proba.max():.4f}]")
if y_proba.max() < 0.9:
    print("   ⚠️  Max probability < 0.9 - Model seems uncertain (could be good or bad)")
if y_proba.min() > 0.01:
    print("   ⚠️  Min probability > 0.01 - No very confident negative predictions")

# Temporal/identifier leakage check
print("\n2️⃣ TEMPORAL/IDENTIFIER LEAKAGE CHECK:")

# Check if we have planet names or identifiers that could leak information
feature_names = X_df.columns.tolist() if hasattr(X_df, 'columns') else []
suspicious_features = [col for col in feature_names if any(keyword in col.lower() 
                      for keyword in ['name', 'id', 'date', 'year', 'discovery', 'ref'])]

if suspicious_features:
    print(f"   ⚠️  Suspicious features found: {suspicious_features}")
    print("   → These could contain identifying information leading to leakage")
else:
    print("   ✅ No obvious identifier features detected")

# Check for features that are too predictive
print("\n3️⃣ FEATURE CORRELATION WITH TARGET:")
if hasattr(X_df, 'columns'):
    # Calculate correlation with target for numerical features
    correlations = []
    for col in X_df.select_dtypes(include=[np.number]).columns:
        try:
            corr = np.corrcoef(X_df[col].fillna(0), y_true)[0, 1]
            if abs(corr) > 0.8:
                correlations.append((col, corr))
        except Exception as e:
            print(f"Could not compute correlation for {col}: {e}")
            continue

    if correlations:
        print("   ⚠️  Highly correlated features (|corr| > 0.8):")
        for feat, corr in correlations:
            print(f"     • {feat}: {corr:.3f}")
        print("   → High correlations could indicate target leakage")
    else:
        print("   ✅ No extremely high correlations detected")

# Cross-validation to detect overfitting
print("\n4️⃣ OVERFITTING DETECTION (K-FOLD CROSS-VALIDATION):")

# Prepare data for CV
cv_scores_auc = []
cv_scores_f1 = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)  # Different random state

print("   Running 5-fold cross-validation...")
for fold, (train_idx, val_idx) in enumerate(cv.split(X, y_true)):
    X_train_cv, X_val_cv = X[train_idx], X[val_idx]
    y_train_cv, y_val_cv = y_true[train_idx], y_true[val_idx]
    
    # Generate predictions on validation fold
    trainer.model.eval()
    with torch.no_grad():
        X_val_tensor = torch.FloatTensor(X_val_cv)
        y_proba_cv = trainer.model(X_val_tensor).squeeze().numpy()
        y_pred_cv = (y_proba_cv >= best_f1_threshold).astype(int)
    
    # Calculate metrics
    auc_cv = roc_auc_score(y_val_cv, y_proba_cv)
    f1_cv = f1_score(y_val_cv, y_pred_cv)
    
    cv_scores_auc.append(auc_cv)
    cv_scores_f1.append(f1_cv)
    
    print(f"     Fold {fold+1}: AUC={auc_cv:.3f}, F1={f1_cv:.3f}")

# Analyze CV results
mean_auc = np.mean(cv_scores_auc)
std_auc = np.std(cv_scores_auc)
mean_f1 = np.mean(cv_scores_f1)
std_f1 = np.std(cv_scores_f1)

print("\n   📊 Cross-Validation Results:")
print(f"     • Mean AUC: {mean_auc:.3f} ± {std_auc:.3f}")
print(f"     • Mean F1:  {mean_f1:.3f} ± {std_f1:.3f}")
print(f"     • Test AUC: {roc_auc:.3f}")
print(f"     • Test F1:  {f1_score(y_test, y_pred_optimal):.3f}")

# Check for overfitting indicators
test_auc = roc_auc_score(y_test, y_proba)
test_f1 = f1_score(y_test, y_pred_optimal)

if test_auc > mean_auc + 2*std_auc:
    print("   ⚠️  Test AUC significantly higher than CV - POSSIBLE OVERFITTING")
elif test_auc < mean_auc - 2*std_auc:
    print("   ⚠️  Test AUC significantly lower than CV - Possible unlucky split")
else:
    print("   ✅ Test AUC consistent with cross-validation")

if test_f1 > mean_f1 + 2*std_f1:
    print("   ⚠️  Test F1 significantly higher than CV - POSSIBLE OVERFITTING")
elif test_f1 < mean_f1 - 2*std_f1:
    print("   ⚠️  Test F1 significantly lower than CV - Possible unlucky split")
else:
    print("   ✅ Test F1 consistent with cross-validation")

# Learning curve analysis (if we have access to training history)
print("\n5️⃣ MODEL COMPLEXITY ANALYSIS:")

# Check model size vs dataset size
n_params = sum(p.numel() for p in trainer.model.parameters())
n_samples = len(X_train)
params_per_sample = n_params / n_samples

print(f"   • Model parameters: {n_params:,}")
print(f"   • Training samples: {n_samples:,}")
print(f"   • Parameters per sample: {params_per_sample:.2f}")

if params_per_sample > 1:
    print("   ⚠️  More parameters than samples - HIGH OVERFITTING RISK")
elif params_per_sample > 0.1:
    print("   ⚠️  Many parameters relative to samples - MODERATE OVERFITTING RISK")
else:
    print("   ✅ Reasonable parameter-to-sample ratio")

# Feature importance analysis for engineered features
print("\n6️⃣ ENGINEERED FEATURES ANALYSIS:")

# Check which engineered features might be too predictive
engineered_features = cfg.ENGINEERED_NUMERIC_FEATURES
available_features = [f for f in engineered_features if f in df_features.columns]

print(f"   Available engineered features: {available_features}")

# Correlation of engineered features with target
for feature in available_features:
    if feature != cfg.TARGET_FEATURE:
        feature_values = df_features[feature].fillna(0).values
        corr = np.corrcoef(feature_values, y_true_raw)[0, 1]
        print(f"   • {feature}: correlation = {corr:.3f}")
        
        if abs(corr) > 0.9:
            print("     ⚠️  VERY HIGH correlation - possible target leakage!")
        elif abs(corr) > 0.7:
            print("     ⚠️  High correlation - check feature engineering")

# Summary and recommendations
print("\n🎯 LEAKAGE/OVERFITTING ASSESSMENT:")

risk_factors = []
if roc_auc > 0.95:
    risk_factors.append("Extremely high ROC-AUC")
if suspicious_features:
    risk_factors.append("Suspicious identifier features")
if params_per_sample > 0.1:
    risk_factors.append("High parameter-to-sample ratio")
if test_auc > mean_auc + 2*std_auc:
    risk_factors.append("Test performance >> CV performance")

if len(risk_factors) >= 3:
    print("   🚨 HIGH RISK of data leakage or overfitting")
    print("   → Investigate feature engineering and data splitting")
elif len(risk_factors) >= 1:
    print("   ⚠️  MODERATE RISK of data leakage or overfitting")
    print("   → Review identified risk factors")
else:
    print("   ✅ LOW RISK of data leakage or overfitting")
    print("   → Model performance appears legitimate")

if risk_factors:
    print("\n   Risk factors identified:")
    for factor in risk_factors:
        print(f"     • {factor}")

print("\n📋 RECOMMENDATIONS:")
print("   1. Use different random states for train/test splits")
print("   2. Implement temporal splits if data has time component")
print("   3. Remove identifier features from model training")
print("   4. Validate on completely new, unseen data")
print("   5. Check feature engineering for target leakage")